In [1]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
# =====================================================================
# CELL 2: IMPORT BATCH PIPELINE
# =====================================================================

from batch_propagation_pipeline import *
ideal_grid_positions = [
    (0.0000, 0.0000), (1.0000, 0.0000), (0.5000, 0.8660), (-0.5000, 0.8660),
    (-1.0000, 0.0000), (-0.5000, -0.8660), (0.5000, -0.8660), (2.0000, 0.0000),
    (1.5000, 0.8660), (1.0000, 1.7321), (0.0000, 1.7321), (-1.0000, 1.7321),
    (-1.5000, 0.8660), (-2.0000, 0.0000), (-1.5000, -0.8660), (-1.0000, -1.7321),
    (0.0000, -1.7321), (1.0000, -1.7321), (1.5000, -0.8660)
]
print("Batch pipeline imported successfully")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Batch pipeline imported successfully


In [3]:

import time
import numpy as np
 
from batch_propagation_pipeline import (
    get_waveguide_properties,
    IncidentFieldGenerator,
    ModalProjector,
    BatchPropagationPipeline,
    DEFAULT_GRID_RESOLUTION,
)
 
 
def profile_pipeline_preset(prop12, p, ifunc, grid_resolution=DEFAULT_GRID_RESOLUTION):
    """
    Manually replays BatchPropagationPipeline.__init__'s steps for one
    wavelength, timing each one. Does NOT construct a real pipeline --
    this is read-only profiling against an already-built prop12.
    """
    timings = {}
 
    t0 = time.perf_counter()
    wvg_props_input = get_waveguide_properties(prop12, mesh_z=0)
    t1 = time.perf_counter()
    timings["get_waveguide_properties(input, mesh_z=0)"] = t1 - t0
 
    wvg_props_output = get_waveguide_properties(prop12, mesh_z=p["z_ex"])
    t2 = time.perf_counter()
    timings["get_waveguide_properties(output, mesh_z=z_ex)"] = t2 - t1
 
    field_gen = IncidentFieldGenerator(p, ifunc, xp=np)
    t3 = time.perf_counter()
    timings["IncidentFieldGenerator.__init__ (mask + ifunc_matrix)"] = t3 - t2
 
    field_gen.precompute_interpolation_weights(wvg_props_input["points"])
    t4 = time.perf_counter()
    timings["precompute_interpolation_weights"] = t4 - t3
 
    modal_proj = ModalProjector(wvg_props_input, xp=np)
    t5 = time.perf_counter()
    timings["ModalProjector.__init__"] = t5 - t4
 
    # Replicates _precompute_delaunay_grid without needing a fully
    # constructed BatchPropagationPipeline instance.
    dummy = BatchPropagationPipeline.__new__(BatchPropagationPipeline)
    dummy.wvg_props_output = wvg_props_output
    _ = dummy._precompute_delaunay_grid(grid_resolution)
    t6 = time.perf_counter()
    timings[f"_precompute_delaunay_grid(resolution={grid_resolution})"] = t6 - t5
 
    print("\n" + "=" * 60)
    print("PRESET-PHASE TIMING BREAKDOWN")
    print("=" * 60)
    total = t6 - t0
    for label, dt in timings.items():
        bar = "#" * max(1, int(40 * dt / max(total, 1e-9)))
        print(f"{dt:7.3f} s  {bar:<40s}  {label}")
    print("-" * 60)
    print(f"{total:7.3f} s  TOTAL")
    print("=" * 60)
 
    return timings

In [4]:
# =====================================================================
# CELL 3: INITIALIZE SYSTEM PARAMETERS
# =====================================================================

p = get_simulation_parameters(3, 0.81)

print("\n" + "="*60)
print("SIMULATION PARAMETERS")
print("="*60)
print(f"Wavelength: {p['wl']} μm")
print(f"Lantern length: {p['z_ex']} μm")
print(f"Cladding radius: {p['rclad']} μm")
print(f"Jacket radius: {p['rjack']} μm")
print(f"Core radius: {p['rcore']:.3f} μm")
print(f"Taper factor: {p['taper_factor']}")
print(f"Available influence function: {p['ifunc_file']}")


SIMULATION PARAMETERS
Wavelength: 0.81 μm
Lantern length: 100000 μm
Cladding radius: 9.0 μm
Jacket radius: 27 μm
Core radius: 0.150 μm
Taper factor: 12.0
Available influence function: /raid2/gcarla/git/ANDES/andes/PASSATA_scripts/data/ifunc/ANDES_400pix_all_modes.fits


In [ ]:
# =====================================================================
# CELL 4: LOAD INFLUENCE FUNCTION AND PROPAGATOR
# =====================================================================

import specula

print("Initializing SPECULA GPU...")
specula.init(0)
from specula.data_objects.ifunc import IFunc

print("Loading influence function...")
# UPDATE THIS LINE TO EXPLICITLY PASS THE DEVICE INDEX:
print(p["ifunc_file"])
ifunc = IFunc.restore(p["ifunc_file"]) 
mask = ifunc.mask_inf_func.get()



# Find the bounding box or centroid
y_cent = np.mean(np.where(mask > 0)[0])
x_cent = np.mean(np.where(mask > 0)[1])
# Desired center (assuming square mask)
center_y = mask.shape[0] // 2
center_x = mask.shape[1] // 2
shift_y = int(center_y - y_cent)
shift_x = int(center_x - x_cent)

print('shift_x, shift_y', shift_x, shift_y)

mask_centered = np.roll(mask, shift_y, axis=0)
mask_centered = np.roll(mask_centered, shift_x, axis=1)
ifunc.mask_inf_func.set(mask_centered)

# Replace the mask
ifunc.mask_inf_func.set(mask_centered)

print(f"  Influence function modes: {len(ifunc.influence_function)}")

print("\nBuilding photonic lantern propagator...")
prop12 = build_and_characterize_lantern(p)

print(f"  Propagator type: {type(prop12).__name__}")
print(f"  Total z range: {prop12.zs[0]:.0f} to {prop12.zs[-1]:.0f} μm")

2026-09-04 14:56:34,870 [INFO]: [specula]: Cupy import successfull. Installed version is: 14.1.1
2026-09-04 14:56:35,202 [INFO]: [specula]: Default device is GPU number 0


Initializing SPECULA GPU...
Loading influence function...
/raid2/gcarla/git/ANDES/andes/PASSATA_scripts/data/ifunc/ANDES_400pix_all_modes.fits
shift_x, shift_y 0 0
  Influence function modes: 10

Building photonic lantern propagator...
mesh has  12495  points
computing modes ...
current z: 36430.0 / 50000 ; current zstep: 640.0              

In [ ]:
# =====================================================================
# CELL 5: CREATE BATCH PROPAGATION PIPELINE
# =====================================================================

print("Creating batch propagation pipeline...\n")
pipeline = BatchPropagationPipeline(prop12, p, ifunc)

diagnose_input_psf(pipeline)

print("Waveguide Properties:")
print(f"  Input modes: {pipeline.wvg_props_input['n_modes']}")
print(f"  Output modes: {pipeline.wvg_props_output['n_modes']}")
print(f"  Mesh points at input: {pipeline.wvg_props_input['points'].shape[0]}")
print(f"  Mesh points at output: {pipeline.wvg_props_output['points'].shape[0]}")

In [ ]:
profile_pipeline_preset(prop12, p, ifunc)

In [ ]:
#coeff1, _labels1 = create_random_aberration_configs(n=1, m=10, minv=-0, maxv=0)
coeff0, _labels0 = create_random_aberration_configs(n=100, m=9, minv=400, maxv=400)
coeff1, labels = create_ramp_aberration_configs([0],20,0,400)

In [ ]:
print("\nGenerating batch of input modal coefficients...\n")
import time
print(coeff0.shape)
start_time = time.time()
u0_batch = pipeline.generate_batch_modal_coefficients(coeff0, use_gpu=False)
elapsed = time.time() - start_time
print(f"\generate_batch_modal_coefficients Complete:")
print(f"  Elapsed time: {elapsed:.1f} seconds")
print(f"Input Modal Coefficient Batch:")
print(f"  Shape: {u0_batch.shape}")
print(f"  Dtype: {u0_batch.dtype}")
print(f"  Memory: {u0_batch.nbytes / 1e6:.1f} MB")

u1_batch = pipeline.generate_batch_modal_coefficients(coeff1)


In [ ]:
# =====================================================================
# CELL 8: PROPAGATE BATCH THROUGH LANTERN
# =====================================================================
import time
start_time = time.time()

print("\nPropagating batch through lantern...")
#uf_batch, zs, us_batch = pipeline.propagate_batch_single(u1_batch)
#uf_batch, zs, us_batch = pipeline.propagate_batch(u1_batch)

elapsed = time.time() - start_time

print(f"\nPropagation Complete:")
print(f"  Elapsed time: {elapsed:.1f} seconds")
start_time = time.time()


#uf_batch, zs, us_batch = pipeline.propagate_batch_single(u0_batch)
uf_batch0, zs0, us_batch0 = pipeline.propagate_batch(u0_batch)
uf_batch1, zs1, us_batch1 = pipeline.propagate_batch(u1_batch)

elapsed = time.time() - start_time

print(f"\nPropagation Complete:")
print(f"  Elapsed time: {elapsed:.1f} seconds")
print(f"  Output modal coefficients shape: {uf_batch0.shape}")
print(f"  Propagation z-points: {len(zs0)}")

In [ ]:
uf_batch0[0]

In [ ]:
# =====================================================================
# CELL 9: RECONSTRUCT OUTPUT SPATIAL FIELDS
# =====================================================================

print("Reconstructing output spatial fields...\n")

E_output_batch0 = pipeline.reconstruct_batch_output_fields(uf_batch0)
E_output_batch1 = pipeline.reconstruct_batch_output_fields(uf_batch1)


print(f"Output Spatial Fields:")
print(f"  Shape: {E_output_batch0.shape}")
print(f"  Dtype: {E_output_batch0.dtype}")
print(f"  Memory: {E_output_batch0.nbytes / 1e6:.1f} MB")


In [ ]:
# =====================================================================
# CELL 10: INTERPOLATE TO REGULAR GRID
# =====================================================================

print("Interpolating output fields to regular grid...\n")

uf_2d_batch0, X_plot0, Y_plot0 = pipeline.interpolate_output_to_grid(E_output_batch0, grid_resolution=200)
uf_2d_batch1, X_plot1, Y_plot1 = pipeline.interpolate_output_to_grid(E_output_batch1, grid_resolution=200)

print(f"Interpolated Intensity Maps:")
print(f"  Shape: {uf_2d_batch0.shape}")
print(f"  X range: [{X_plot0.min():.2f}, {X_plot0.max():.2f}] μm")
print(f"  Y range: [{Y_plot0.min():.2f}, {Y_plot0.max():.2f}] μm")
print(f"  Memory: {uf_2d_batch0.nbytes / 1e6:.1f} MB")

In [ ]:
# =====================================================================
# CELL 11: VISUALIZE BATCH OUTPUT
# =====================================================================

print("Generating visualization...\n")

# Create titles for each field
titles = [] #[f"Mode {c['mode_idx']}, {c['amplitude_nm']:.0f} nm" 
          #for c in aberration_configs]

# Visualize
visualize_batch_output(uf_2d_batch1[0:2], X_plot1, Y_plot1, titles, maxv=1, show_arrow=False)

In [ ]:
# --- Step 1: Detect core centers from the first output field ---
first_intensity = uf_2d_batch0.sum(axis=0)
# shape (grid_resolution, grid_resolution)
# But we need the spatial coordinates X_plot, Y_plot from the pipeline.
# Actually, we have X_plot, Y_plot from the interpolation.

# Reuse your calibration function, but now pass the 2D intensity map and the mesh points?
# The calibration function expects a (N_points,) profile and mesh.points.
# But we already have the interpolated grid. Let's create a function that works directly on grid.


# Detect centers from the first output field
core_centers = detect_centers_from_grid(first_intensity, X_plot0, Y_plot0)
print(f"Detected {len(core_centers)} core centers")

In [ ]:

# Detect centers once
x_min, y_min = X_plot0[0,0], Y_plot0[0,0]
dx = X_plot0[0,1] - X_plot0[0,0]
dy = Y_plot0[1,0] - Y_plot0[0,0]

core_centers = detect_centers_from_grid(first_intensity, X_plot0, Y_plot0)
ideal_permutation = map_evaluated_to_ideal_geometry(core_centers, ideal_grid_positions)


In [ ]:

for i in range(len(uf_2d_batch1)):
    # Extract signals for field i
    signals = collect_subpixel_signals(uf_2d_batch1[i], x_min, y_min, dx, dy, core_centers)
    standardized = np.zeros(19)
    standardized[ideal_permutation] = signals*10
    display_hex_grid_plots(ideal_grid_positions, standardized)

In [ ]:
# Batch sub-pixel centroid tracking + geometric hex mapping, then a hex-grid
# plot per field.  calibrate_subpixel_centers / map_evaluated_to_ideal_geometry
# / display_hex_grid_plots come from the batch_pipeline package (already in
# scope via `from batch_propagation_pipeline import *`); the local copies that
# used to be redefined here had drifted from the package versions (e.g. the
# Procrustes reflection guard was missing) and the call still used the old
# "pass every helper in as an argument" signature.
titles = []
visualize_batch_hex_grid_signals(
    pipeline,
    E_output_batch1[:5],
    ideal_grid_positions,
    titles=titles,
    grid_resolution=400,
)


In [ ]:
# Same call on a smaller slice.
visualize_batch_hex_grid_signals(
    pipeline,
    E_output_batch1[:2],
    ideal_grid_positions,
    titles=titles,
    grid_resolution=400,
)


In [ ]:
# =====================================================================
# CELL 12: BATCH STATISTICS AND ANALYSIS
# =====================================================================

print("\nComputing batch statistics...\n")

batch_statistics(uf_batch, titles)

# Additional analysis: Mode coupling
print("\n" + "="*60)
print("MODE COUPLING ANALYSIS")
print("="*60)

for i in [0, 10, -1]:
    config = aberration_configs[i]
    mode_powers = np.abs(uf_batch[i])**2
    top_3_indices = np.argsort(mode_powers)[-3:][::-1]
    
    print(f"\n{titles[i]}:")
    print(f"  Top 3 output modes: {top_3_indices}")
    print(f"  Their powers: {mode_powers[top_3_indices]}")
    print(f"  Concentrated power: {np.sum(mode_powers[top_3_indices]):.4f}")

In [ ]:
# =====================================================================
# CELL 13: EXTRACT AND ANALYZE SPECIFIC METRICS
# =====================================================================

# Compute power in different mode groups
print("\n" + "="*60)
print("POWER DISTRIBUTION ANALYSIS")
print("="*60)

def analyze_power_distribution(uf_batch):
    """Analyze how power distributes among modes."""
    n_fields = uf_batch.shape[0]
    
    fundamental_power = np.abs(uf_batch[:, 0])**2
    top_3_power = np.sum(np.abs(uf_batch[:, :3])**2, axis=1)
    top_5_power = np.sum(np.abs(uf_batch[:, :5])**2, axis=1)
    total_power = np.sum(np.abs(uf_batch)**2, axis=1)
    
    return {
        'fundamental': fundamental_power,
        'top_3': top_3_power,
        'top_5': top_5_power,
        'total': total_power
    }

power_dist = analyze_power_distribution(uf_batch)

print(f"\nFundamental Mode Power:")
print(f"  Mean: {np.mean(power_dist['fundamental']):.4f}")
print(f"  Std: {np.std(power_dist['fundamental']):.4f}")
print(f"  Min: {np.min(power_dist['fundamental']):.4f}")
print(f"  Max: {np.max(power_dist['fundamental']):.4f}")

print(f"\nTop 3 Modes Power:")
print(f"  Mean: {np.mean(power_dist['top_3']):.4f}")
print(f"  Std: {np.std(power_dist['top_3']):.4f}")

In [ ]:
# =====================================================================
# CELL 14: SAVE RESULTS FOR POST-PROCESSING
# =====================================================================

import pickle
from pathlib import Path

# Create results directory
results_dir = Path('batch_results')
results_dir.mkdir(exist_ok=True)

# Save data
results = {
    'u0_batch': u0_batch,
    'uf_batch': uf_batch,
    'E_output_batch': E_output_batch[:10],
    'uf_2d_batch': uf_2d_batch,
    'zs': zs,
    'us_batch': us_batch,
    'configs': aberration_configs,
    'X_plot': X_plot,
    'Y_plot': Y_plot
}

# Save as pickle
with open(results_dir / 'batch_results.pkl', 'wb') as f:
    pickle.dump(results, f)

# Also save numpy arrays separately for easier access
np.save(results_dir / 'u0_batch.npy', u0_batch)
np.save(results_dir / 'uf_batch.npy', uf_batch)
np.save(results_dir / 'uf_2d_batch.npy', uf_2d_batch)

print(f"Results saved to {results_dir}/")
print(f"  - batch_results.pkl (complete results)")
print(f"  - u0_batch.npy (input modal coefficients)")
print(f"  - uf_batch.npy (output modal coefficients)")
print(f"  - uf_2d_batch.npy (output intensity maps)")

In [ ]:
# =====================================================================
# CELL 15: MODE EVOLUTION ALONG PROPAGATION
# =====================================================================

# Analyze how modes evolve during propagation
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Select 4 representative fields
selected_indices = [0, len(aberration_configs)//3, 2*len(aberration_configs)//3, -1]

for ax, idx in zip(axes.flat, selected_indices):
    # us_batch shape: (n_fields, n_z, n_modes)
    # We want the evolution of the top 3 modes for field idx
    
    evolution = us_batch[idx]  # shape: (n_z, n_modes)
    
    # Find top 3 output modes
    output_powers = np.abs(uf_batch[idx])**2
    top_modes = np.argsort(output_powers)[-3:][::-1]
    
    # Plot evolution
    for mode_idx in top_modes:
        mode_evolution = np.abs(evolution[:, mode_idx])**2
        ax.plot(zs, mode_evolution, label=f'Mode {mode_idx}')
    
    ax.set_xlabel('z (μm)')
    ax.set_ylabel('Power')
    ax.set_title(titles[idx])
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Mode Power Evolution During Propagation')
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# CELL 16: EXPORT TO HDF5 FOR EXTERNAL ANALYSIS
# =====================================================================

try:
    import h5py
    
    filename = results_dir / 'batch_results.h5'
    
    with h5py.File(filename, 'w') as f:
        # Input/output modal coefficients
        f.create_dataset('u0_batch', data=u0_batch)
        f.create_dataset('uf_batch', data=uf_batch)
        
        # Evolution
        f.create_dataset('zs', data=zs)
        f.create_dataset('us_batch', data=us_batch)
        
        # Grid data
        f.create_dataset('X_plot', data=X_plot)
        f.create_dataset('Y_plot', data=Y_plot)
        f.create_dataset('uf_2d_batch', data=uf_2d_batch)
        
        # Metadata
        f.attrs['n_fields'] = len(aberration_configs)
        f.attrs['n_modes'] = uf_batch.shape[1]
        f.attrs['n_z_points'] = len(zs)
        f.attrs['wavelength_um'] = p['wl']
        f.attrs['lantern_length_um'] = p['z_ex']
    
    print(f"HDF5 file saved: {filename}")
except ImportError:
    print("h5py not installed. Skipping HDF5 export.")